# GAS-SN sweep regressions — one cell per slide

Every calculation behind `slides/frac_slides.md`. Each code cell starts with its slide
title and prints the tables and numbers that slide quotes, so any figure on a slide can
be traced to the code and the CSV that produced it. Pure-text/theory slides are skipped.

Run from `slides/`.

## How to read the statistics

**The two-term law** `threshold = a + b*kappa`. The *threshold* is how far apart the two
regimes must sit (in MAD units) before a model reaches `TARGET` accuracy. `a` is what it
needs at Gaussian tails; `b` is what each unit of excess kurtosis costs.

**R² of that law** says how completely a single tail number (kappa) predicts what a model
needs -- i.e. how *threshold-like* the model is. High R² = a clean law.

**F-test (adding alpha, k to a law in kappa alone)**

$$F \;=\; \frac{\Delta R^2 / q}{(1 - R^2_{\mathrm{full}})/(n-p)}$$

that is, **improvement per added parameter** divided by **leftover noise per residual dof**.

`p` is that F read off the F(q, n-p) distribution: the chance of an improvement this large
if alpha and k truly had **zero** effect. So bigger F and smaller p are the *same statement*,
not two.

> **F measures evidence, not importance.** A large F can come from a tiny improvement
> against tiny residual noise. Read `dR2` for the size of the effect and `F`/`p` for how
> sure we are it is real. The cells below print both, plus the improvement as a share of
> what kappa left unexplained -- which is the number that makes the models comparable.

**t-statistics** in a multi-term regression are partial: the effect of that variable with
the others held fixed. |t| > 2 is the usual "distinguishable from zero" line.

In [ ]:
# === Setup: shared helpers used by every slide cell ===
import sys, os
import numpy as np, pandas as pd, statsmodels.formula.api as smf
sys.path.insert(0, os.path.abspath('..'))   # notebook lives in slides/; repo root is ..
from compare.threshold_law import crossing

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
TARGET = 0.65        # every law in the deck is quoted at this target

def threshold_cells(d, keys, target=TARGET):
    """One row per `keys` cell: cluster distance (MAD units) where the replicate-mean
    bac first reaches `target`, plus the excess kurtosis there.

    crossing() returns None in TWO cases -- bac never reaches target (too hard) and bac
    is above it throughout (too easy) -- so cells are dropped from both ends.
    """
    keys = list(keys); out = []
    for vals, g in d.groupby(keys):
        c = (g.groupby('loc')
               .agg(bac=('bac', 'mean'), cd=('cdist_mad', 'mean'), ku=('kurt', 'mean'))
               .sort_index())
        xs = c.index.values.astype(float)
        t = crossing(xs, c['bac'].values, target)
        if t is None:
            continue
        rec = dict(zip(keys, vals if isinstance(vals, tuple) else (vals,)))
        rec['thr']   = float(np.interp(t, xs, c['cd'].values))
        rec['kappa'] = float(np.interp(t, xs, c['ku'].values))
        out.append(rec)
    return pd.DataFrame(out)

def fit_two_term_law(cells):
    """Fit the two-term law  threshold = a + b*kappa  ->  (a, b, R2, n)."""
    x, y = cells['kappa'].values, cells['thr'].values
    A = np.vstack([np.ones_like(x), x]).T
    b, *_ = np.linalg.lstsq(A, y, rcond=None)
    r = y - A @ b
    return b[0], b[1], 1 - r @ r / ((y - y.mean()) @ (y - y.mean())), len(x)

VAE_SWEEP  = pd.read_csv('../data/vae_sweep.csv')          # 3 models, clip 8/12/20
CLIP18     = pd.read_csv('../data/vae_sweep_clip18.csv')   # 3 models, clip 18
JUMP_SWEEP = pd.read_csv('../data/jump_sweep.csv')         # jump(+kmeans), 5 penalties
KM_SWEEP   = pd.read_csv('../data/kmeans_sweep.csv')       # kmeans, 4 feature sets

for n, d in [('vae_sweep', VAE_SWEEP), ('vae_sweep_clip18', CLIP18),
             ('jump_sweep', JUMP_SWEEP), ('kmeans_sweep', KM_SWEEP)]:
    print(f"{n:18s} {len(d):6d} rows   models={sorted(d['model'].unique())}")


In [ ]:
# === Model efficacy — The two-term law for cluster distance and tail weight ===
# Slide quotes: R2 = 0.99 Jump / 0.78 VAE / 0.65 KMeans (within a config);
#               pooling over configs drops R2 to 0.48 (jump) and 0.56 (all nine);
#               alpha and k do NOT vanish into kappa (F-tests).
c12 = threshold_cells(VAE_SWEEP[VAE_SWEEP['clip_factor'] == 12], ['model', 'alpha', 'k'])
print("law within clip=12 (data/vae_sweep.csv) -- the R2 the slide quotes:")
for m in ('vae', 'jump', 'kmeans'):
    a, b, r2, n = fit_two_term_law(c12[c12['model'] == m])
    print(f"  {m:7s}  {a:.3f} + {b:.4f}*kappa   R2={r2:.2f}  (n={n} cells)")

nine = threshold_cells(VAE_SWEEP, ['model', 'clip_factor', 'alpha', 'k'])
a, b, r2, n = fit_two_term_law(nine)
print(f"\npooled over all nine (model, clip) configs: R2={r2:.2f}  (n={n})")
jp = threshold_cells(JUMP_SWEEP[JUMP_SWEEP['model'] == 'jump'],
                     ['clip_factor', 'jump_penalty', 'alpha', 'k'])
a, b, r2, n = fit_two_term_law(jp)
print(f"jump pooled over clip x penalty (data/jump_sweep.csv): R2={r2:.2f}  (n={n})")

# Is kappa a SUFFICIENT statistic for the tail, or do the generating parameters
# alpha and k still carry information once kappa is known?  Nested F-test:
#     restricted   thr ~ kappa
#     full         thr ~ kappa + alpha + k
# F = (dR2 / 2) / ((1 - R2_full) / (n - 4)).  Big F = the gain is large RELATIVE TO the
# noise still left -- which is NOT the same as the gain being large.  `dR2` is the size
# of the effect; `of remainder` is dR2 as a share of what kappa left unexplained, and is
# the fairest cross-model comparison because it corrects for how good the baseline was.
print("\ndoes adding alpha,k beat the law in kappa alone?  (all 4 clips pooled)")
allc = pd.concat([threshold_cells(VAE_SWEEP, ['model', 'clip_factor', 'alpha', 'k']),
                  threshold_cells(CLIP18,    ['model', 'clip_factor', 'alpha', 'k'])],
                 ignore_index=True)
print(f"  {'model':7s} {'n':>3s} {'R2 kappa':>9s} {'R2 +a,k':>8s} {'dR2':>7s} "
      f"{'F':>7s} {'p':>9s} {'of remainder':>13s}")
for m in ('jump', 'vae', 'kmeans'):
    s = allc[allc['model'] == m]
    m1 = smf.ols('thr ~ kappa', data=s).fit()
    m2 = smf.ols('thr ~ kappa + alpha + k', data=s).fit()
    F, pv, _ = m2.compare_f_test(m1)
    dR2 = m2.rsquared - m1.rsquared
    print(f"  {m:7s} {len(s):3d} {m1.rsquared:9.3f} {m2.rsquared:8.3f} {dR2:7.3f} "
          f"{F:7.2f} {pv:9.1e} {dR2/(1-m1.rsquared):12.0%}")
print("  Jump: strong evidence (p=0.0015) of a TINY effect -- dR2 is 0.009, but the")
print("        residual it is measured against is only 0.026.")
print("  KMeans: strong evidence of a LARGE effect -- alpha,k more than double the R2.")
print("  Note the F ranking (kmeans > vae > jump) REVERSES the R2(kappa) ranking:")
print("  R2 ranks how threshold-like a model is; F ranks how badly kappa alone")
print("  summarises the tail for it.  p rejects 'no effect' for all three.")


In [ ]:
# === The two-term law: The intercept transfers, the slope does not ===
# Slide quotes: a = 0.313 +- 0.021 (1.2x spread), b = 0.057 +- 0.024 (4.1x);
#               b by clip = 0.081 / 0.057 / 0.035;
#               penalty sweep: 18 configs, a = 0.277 +- 0.034, b = 0.062 +- 0.028;
#               target sensitivity: a runs 0.24 -> 0.44 over target 0.58 -> 0.72.
rows = []
for (m, cl), g in VAE_SWEEP.groupby(['model', 'clip_factor']):
    a, b, r2, n = fit_two_term_law(threshold_cells(g, ['alpha', 'k']))
    rows.append(dict(model=m, clip=cl, a=a, b=b, R2=r2, n=n))
nine = pd.DataFrame(rows)
print("the nine laws (data/vae_sweep.csv, target 0.65):")
print(nine.round(4).to_string(index=False))
print(f"\n  a = {nine['a'].mean():.3f} +- {nine['a'].std(ddof=1):.3f}"
      f"   spread {nine['a'].max()/nine['a'].min():.1f}x")
print(f"  b = {nine['b'].mean():.3f} +- {nine['b'].std(ddof=1):.3f}"
      f"   spread {nine['b'].max()/nine['b'].min():.1f}x")
print("  b by clip:", nine.groupby('clip')['b'].mean().round(4).to_dict())

rows = []
for keys, g in JUMP_SWEEP.groupby(['model', 'clip_factor', 'jump_penalty']):
    L = fit_two_term_law(threshold_cells(g, ['alpha', 'k'])) if len(g) else None
    if L and L[3] >= 3:
        rows.append(dict(zip(['model', 'clip', 'penalty'], keys)) | dict(a=L[0], b=L[1]))
cfg = pd.DataFrame(rows)
print(f"\npenalty sweep (data/jump_sweep.csv): {len(cfg)} configs"
      f" -- {(cfg['model']=='jump').sum()} jump + {(cfg['model']=='kmeans').sum()} penalty-invariant kmeans")
print(f"  a = {cfg['a'].mean():.3f} +- {cfg['a'].std(ddof=1):.3f}"
      f"   b = {cfg['b'].mean():.4f} +- {cfg['b'].std(ddof=1):.4f}")

sub = JUMP_SWEEP[(JUMP_SWEEP['model'] == 'jump') & (JUMP_SWEEP['clip_factor'] == 12)
                 & (JUMP_SWEEP['jump_penalty'] == 100)]
tg_rows = [(tg, fit_two_term_law(threshold_cells(sub, ['alpha', 'k'], target=tg)))
           for tg in (0.58, 0.60, 0.65, 0.70, 0.72)]
print("\nthe same law re-read at other targets (jump, clip 12, penalty 100):\n  " + "   ".join(
    f"t={tg:.2f}: a={L[0]:.3f} b={L[1]:.4f}" for tg, L in tg_rows)
      + "\n  -> `a` moves a lot with the target, `b` barely: a law needs its target quoted.")


In [ ]:
# === `clip` is not a setting - it selects the (alpha, k, kappa) region you sample ===
# Slide quotes: the 4-term regression (kappa +0.018 t=+4.8 / alpha -0.200 t=-6.0 /
#               k -0.035 t=-5.4 / clip -0.002 p=0.32); effect sizes -0.160/-0.141/+0.165;
#               the kappa^2 control; the kappa a clip reaches; the clip-18 falsification.
#
# HOW TO READ IT: each t is a PARTIAL effect -- that variable with the other three held
# fixed.  clip is the only term that fails |t|>2, so once you know (alpha, k, kappa),
# clip tells you nothing more: it is not a knob the law responds to, it merely decides
# which (alpha, k, kappa) cells the sweep can reach.  The kappa^2 block is the check that
# alpha and k are not just standing in for curvature in kappa -- if they were, adding
# kappa^2 would absorb them.  Instead kappa itself collapses and alpha,k get STRONGER.
CELLS140 = pd.concat([threshold_cells(VAE_SWEEP, ['model', 'clip_factor', 'alpha', 'k']),
                      threshold_cells(CLIP18,    ['model', 'clip_factor', 'alpha', 'k'])],
                     ignore_index=True).rename(columns={'clip_factor': 'clip'})
print(f"{len(CELLS140)} threshold cells; {144 - len(CELLS140)} of 144 did not cross "
      f"({100*(144-len(CELLS140))/144:.1f}% dropped -> survivorship is small)")

res = smf.ols('thr ~ kappa + alpha + k + clip + C(model)', data=CELLS140).fit()
print(f"\nthr ~ kappa + alpha + k + clip + C(model)     n={int(res.nobs)}  R2={res.rsquared:.3f}")
print(f"  {'term':22s} {'coef':>9s} {'t':>7s} {'p':>10s}")
for c in res.params.index:          # every term, intercept and model dummies included
    print(f"  {c:22s} {res.params[c]:+9.5f} {res.tvalues[c]:+7.2f} {res.pvalues[c]:10.2e}")

# The Intercept here is NOT the two-term law's `a`.  It is the fitted value at
# kappa=0 AND alpha=0, k=0, clip=0 -- and alpha, k, clip are swept only over
# [0.8,1.6], [2,6], [8,20], so it extrapolates far outside the grid and is not
# physically meaningful on its own.  C(model) drops one level, so the Intercept is
# that reference model's level and the other C(model)[T.*] rows are offsets from it.
def pred_at(fit, m, ka, al, kk_, cl):
    """Fitted threshold at one (kappa, alpha, k, clip) point for model m."""
    return fit.predict(pd.DataFrame([dict(kappa=ka, alpha=al, k=kk_, clip=cl, model=m)])).iloc[0]

ref = [l for l in CELLS140['model'].unique() if f'C(model)[T.{l}]' not in res.params.index][0]
at = {c: CELLS140[c].mean() for c in ('alpha', 'k', 'clip')}
gauss = {m: res.predict(pd.DataFrame([{**at, 'kappa': 0.0, 'model': m}])).iloc[0]
         for m in ('vae', 'jump', 'kmeans')}
print(f"\n  Intercept absorbs model='{ref}'; it sits at alpha=k=clip=0, far OUTSIDE the swept grid,"
      f" so it is not the law's `a`.")
print(f"  at kappa=0 with alpha={at['alpha']:.2f}, k={at['k']:.2f}, clip={at['clip']:.1f} (grid means):   "
      + "   ".join(f"{m}: {v:.3f}" for m, v in gauss.items())
      + "   [deck's two-term a: 0.333/0.311/0.281]")
# WHY is the Intercept so much bigger than those?  Walk each term from 0 up to its mean.
# It is NOT clip's scale: clip has the widest span but by far the smallest coefficient,
# and only the PRODUCT matters.  alpha=0 and k=0 are not merely off-grid, they are
# physically nonexistent (alpha is the stability index, swept 0.8-1.6), so the Intercept
# extrapolates to a configuration that cannot occur.  kappa=0 IS real (Gaussian tails),
# which is why the two-term law's `a` is interpretable and this Intercept is not.
walk = {v: res.params[v] * CELLS140[v].mean() for v in ('alpha', 'k', 'clip')}
gap = res.params['Intercept'] - gauss['jump']
print(f"  walking jump from Intercept {res.params['Intercept']:.3f} to {gauss['jump']:.3f}:   "
      + "   ".join(f"{v}->mean {d:+.3f} ({100*-d/gap:.0f}%)" for v, d in walk.items()))
print("    alpha dominates; clip is negligible despite the widest range -- only coef x range counts")
# CENTERING (alpha-1 is Cauchy and inside the swept range; k-2 the variance boundary;
# clip-12 the default) moves ONLY the intercept -- every slope, t, p and R2 is identical:
cen = smf.ols('thr ~ kappa + I(alpha-1) + I(k-2) + I(clip-12) + C(model)', data=CELLS140).fit()
print(f"  centered on (alpha-1, k-2, clip-12): Intercept {cen.params['Intercept']:.4f} "
      f"vs raw {res.params['Intercept']:.4f}, R2 {cen.rsquared:.4f} vs {res.rsquared:.4f} -- same fit")
# ...but kappa is a FUNCTION of (alpha,k,clip), so they cannot be centred independently.
# Centring each on its own canonical value gives a point that CANNOT EXIST:
kk = CELLS140[(CELLS140['k'] == 2.0) & (CELLS140['alpha'] == 1.1)]['kappa']
thin = CELLS140[(CELLS140['alpha'] == 1.6) & (CELLS140['k'] == 6.0)]['kappa']
print(f"  but kappa=0 with alpha=1,k=2 cannot occur -- that corner generates kappa "
      f"{kk.min():.1f}-{kk.max():.1f}.  Note b_kappa>0, so centring kappa HIGHER raises the")
print(f"  intercept ({pred_at(res,'jump',0,1,2,12):.3f} -> {pred_at(res,'jump',5,1,2,12):.3f} at kappa=5), it does not lower it.")
print(f"  The SELF-CONSISTENT centre is the thin-tail corner alpha=1.6, k=6, where the data show"
      f" kappa={thin.mean():.2f} ~ 0.")
_deck = {'vae': 0.333, 'jump': 0.311, 'kmeans': 0.281}
print("  there the pooled model reproduces the two-term law's `a`:   " + "   ".join(
    f"{m}: {pred_at(res,m,0,1.6,6.0,12):.3f} (deck {_deck[m]:.3f})" for m in ('vae','jump','kmeans')))

print(f"\neffect over each swept range:   alpha 0.8->1.6 {res.params['alpha']*0.8:+.3f} MAD"
      f"   |   k 2->6 {res.params['k']*4:+.3f} MAD   |   kappa 1->10 {res.params['kappa']*9:+.3f} MAD")

q = smf.ols('thr ~ kappa + I(kappa**2) + alpha + k + C(model)', data=CELLS140).fit()
print("\nkappa^2 control (are alpha,k just curvature?):  " + "   ".join(
    f"{c.replace(' ','')} t={q.tvalues[c]:+.2f}" for c in ('kappa','I(kappa ** 2)','alpha','k'))
      + "   -> kappa collapses, alpha/k strengthen")

# kappa reach is a property of EVERY fit, not only of the cells that crossed target --
# computing it on threshold cells only would understate the tails the sweep actually saw.
ALLROWS = pd.concat([VAE_SWEEP, CLIP18], ignore_index=True)
def _reach(g):
    ku = g['kurt']; return f"{ku.median():.2f}/{ku[ku >= ku.quantile(0.95)].mean():.2f}/{ku.max():.2f}"
rch = [f"clip {cl:.0f}: {_reach(g)}" for cl, g in ALLROWS.groupby('clip_factor')]
print("\nkappa a clip reaches, over all fits (median/top-5%/max):")
print("  " + "   ".join(rch) + f"   |   whole: {_reach(ALLROWS)}  (deck 3.0/15.9/18.6)")

c18 = CELLS140[CELLS140['clip'] == 18]
bs = [fit_two_term_law(c18[c18['model'] == m]) for m in ('vae', 'jump', 'kmeans')]
print("\nclip-18 falsification (predicted b~0.038, a~0.32 BEFORE the run):")
print("  " + "   ".join(f"{m}: {L[0]:.4f}+{L[1]:.4f}k (R2={L[2]:.2f})"
                        for m, L in zip(('vae','jump','kmeans'), bs))
      + f"   -> mean a={np.mean([L[0] for L in bs]):.4f}, b={np.mean([L[1] for L in bs]):.4f} (match)")


In [ ]:
# === `clip` is not a setting - CENTRED on the self-consistent corner ===
# The raw intercept (+0.875) sits at alpha = k = clip = 0, far outside the swept grid, so it
# is not the law's `a`. Centring fixes that -- but the three regressors cannot be centred
# INDEPENDENTLY, because kappa is a function of (alpha, k, clip): at alpha=1, k=2 the data
# show kappa ~ 3-12, never 0, so that point cannot occur.
#
# The self-consistent choice is the THIN-TAIL corner alpha=1.6, k=6, clip=12, where the data
# show kappa ~ 0.5 -- there kappa = 0 is a genuine limit, and the intercept becomes the
# Gaussian-tail threshold, directly comparable with the two-term law's `a`.
A0, K0, C0 = 1.6, 6.0, 12.0
print(f"kappa observed at (alpha={A0}, k={K0}): "
      f"{CELLS140[(CELLS140['alpha'] == A0) & (CELLS140['k'] == K0)]['kappa'].mean():.2f}"
      f"   -> kappa=0 is a real limit here")

cen = smf.ols(f'thr ~ kappa + I(alpha-{A0}) + I(k-{K0}) + I(clip-{C0}) + C(model)',
              data=CELLS140).fit()
print(f"\ncentred on (alpha={A0}, k={K0}, clip={C0})   n={int(cen.nobs)}  R2={cen.rsquared:.4f}")
print(f"  {'term':22s} {'coef':>9s} {'t':>7s} {'p':>10s}")
for c in cen.params.index:
    print(f"  {c:22s} {cen.params[c]:+9.4f} {cen.tvalues[c]:+7.2f} {cen.pvalues[c]:10.2e}")

# centring moves ONLY the intercept
same = np.allclose(sorted(np.abs(res.tvalues.drop('Intercept'))),
                   sorted(np.abs(cen.tvalues.drop('Intercept'))))
print(f"\n  raw Intercept {res.params['Intercept']:+.4f}  ->  centred {cen.params['Intercept']:+.4f}")
print(f"  R2 {res.rsquared:.4f} vs {cen.rsquared:.4f}; slopes and t-stats identical: {same}")

_deck = {'vae': 0.333, 'jump': 0.311, 'kmeans': 0.281}
print("\n  Gaussian-tail threshold at this corner, vs the two-term law's `a`:")
for m in ('vae', 'jump', 'kmeans'):
    print(f"    {m:7s} {pred_at(res, m, 0.0, A0, K0, C0):.3f}   (law a = {_deck[m]:.3f})")
print("  The pooled four-term fit and the two-term law agree once the point is possible.")


In [ ]:
# === Why the slope never transferred - the two-term law is a projection ===
# Slide quotes: the OVB table (delta_alpha, delta_k, predicted vs observed b per clip)
#               and that b_kappa shows no trend while marginal b falls monotonically.
#
# OMITTED-VARIABLE BIAS.  Regressing on kappa alone leaves alpha and k in the error term.
# OLS then gives b = Cov(thr,kappa)/Var(kappa); substituting the true three-term model,
#     b_marginal = b_kappa + b_alpha*delta_alpha + b_k*delta_k
# where delta_x = Cov(x,kappa)/Var(kappa) is the slope of the OMITTED variable regressed
# ON the kept one.  That direction looks backwards -- alpha causes kappa, not the reverse
# -- but it is an algebraic projection onto the regressor you kept, not a causal claim.
# delta is a property of the SWEEP GRID (how the design lands in kappa), not of the law.
st = smf.ols('thr ~ kappa + alpha + k + C(model)', data=CELLS140).fit()
bk, ba, bkk = st.params['kappa'], st.params['alpha'], st.params['k']
print(f"pooled three-term law: {bk:+.5f}*kappa {ba:+.5f}*alpha {bkk:+.5f}*k\n")
print("b_marginal = b_kappa + b_alpha*delta_alpha + b_k*delta_k")
print(f"  {'clip':>5s} {'d_alpha':>9s} {'d_k':>8s} {'predicted':>11s} {'observed':>10s} {'err':>8s}")
for cl, s in CELLS140.groupby('clip'):
    da = smf.ols('alpha ~ kappa', data=s).fit().params['kappa']
    dk = smf.ols('k ~ kappa',     data=s).fit().params['kappa']
    pred = bk + ba * da + bkk * dk
    obs  = smf.ols('thr ~ kappa + C(model)', data=s).fit().params['kappa']
    print(f"  {cl:5.0f} {da:9.4f} {dk:8.4f} {pred:11.4f} {obs:10.4f} {pred-obs:+8.4f}")

print("\nfitted WITHIN each clip the identity is exact algebra (error ~1e-16):")
for cl, s in CELLS140.groupby('clip'):
    f = smf.ols('thr ~ kappa + alpha + k + C(model)', data=s).fit()
    da = smf.ols('alpha ~ kappa + C(model)', data=s).fit().params['kappa']
    dk = smf.ols('k ~ kappa + C(model)',     data=s).fit().params['kappa']
    pred = f.params['kappa'] + f.params['alpha'] * da + f.params['k'] * dk
    obs  = smf.ols('thr ~ kappa + C(model)', data=s).fit().params['kappa']
    print(f"  clip {cl:2.0f}: b_kappa={f.params['kappa']:+.4f}   predicted-observed = {pred-obs:+.2e}")

print("\nwhy delta shrinks:  delta_alpha = corr(alpha,kappa) * sd(alpha)/sd(kappa)")
for cl, s in CELLS140.groupby('clip'):
    c, sa, sk = s['alpha'].corr(s['kappa']), s['alpha'].std(), s['kappa'].std()
    print(f"  clip {cl:2.0f}: corr={c:+.3f}  sd(alpha)={sa:.3f}  sd(kappa)={sk:.3f}  -> {c*sa/sk:+.4f}")


In [ ]:
# === Why MAD, not sd - the ruler the whole framework rests on ===
# Slide quotes: R2 of the two-term law under each ruler, clip-invariance, Spearman
#               rankings, and the Jump law 0.312 + 0.0594*kappa (R2=0.987).
# NOTE on columns: in data/jump_moments.csv, `cdist` is the LEGACY NAME for `cdist_mad`
# (both are d_med/mad -- verified below), not an sd-ruler. The generating notebook defined
# `cdist = d_med/mad` and kept the sd variant separately as `cdist_sd`, which was never
# written to this file. Nothing reads `cdist`. So the sd ruler is derived here as d_mean/sd.
mom = pd.read_csv('../data/jump_moments.csv')
bac = pd.read_csv('../data/jump_loc_sweep.csv')
print(f"cdist is the legacy alias of cdist_mad (both d_med/mad): "
      f"{np.allclose(mom['cdist'], mom['cdist_mad'])}")
print(f"  cdist == d_mean/sd ? {np.allclose(mom['cdist'], mom['d_mean']/mom['sd'])}"
      f"   -> the sd ruler was never stored; deriving it as d_mean/sd")
j = bac.merge(mom, on=['loc', 'alpha', 'k', 'rep'])
j['cd_mad'] = j['d_med'] / j['mad']
j['cd_sd']  = j['d_mean'] / j['sd']

def law_ruler(d, col):
    pts = []
    for (a, k), g in d.groupby(['alpha', 'k']):
        c = g.groupby('loc').agg(bac=('bac', 'mean'), cd=(col, 'mean'), ku=('kurt', 'mean')).sort_index()
        xs = c.index.values.astype(float)
        t = crossing(xs, c['bac'].values, TARGET)
        if t is None: continue
        pts.append((float(np.interp(t, xs, c['ku'].values)), float(np.interp(t, xs, c['cd'].values))))
    x, y = np.array(pts).T
    A = np.vstack([np.ones_like(x), x]).T
    b, *_ = np.linalg.lstsq(A, y, rcond=None); r = y - A @ b
    return b[0], b[1], 1 - r @ r / ((y - y.mean()) @ (y - y.mean())), len(x)

print("\nthe two-term law under each ruler (1,760-fit sweep):")
for name, col in [('median / MAD', 'cd_mad'), ('mean / sd', 'cd_sd')]:
    a, b, r2, n = law_ruler(j, col)
    print(f"  {name:14s} {a:.3f} + {b:.4f}*kappa   R2={r2:.3f}  (n={n})")

print("\nas a single ranker of bac (Spearman, per-cell means):")
cm = j.groupby(['loc', 'alpha', 'k']).agg(bac=('bac', 'mean'), mad=('cd_mad', 'mean'),
                                          sd=('cd_sd', 'mean')).reset_index()
for name, col in [('median / MAD', 'mad'), ('mean / sd', 'sd'), ('raw loc', 'loc')]:
    print(f"  {name:14s} {cm[col].corr(cm['bac'], method='spearman'):.2f}")

print("\nclip-invariance of b, both rulers (data/jump_sweep.csv, jump, penalty 100):")
js = JUMP_SWEEP[(JUMP_SWEEP['model'] == 'jump') & (JUMP_SWEEP['jump_penalty'] == 100)].copy()
js['cd_mad'] = js['d_med'] / js['mad']
js['cd_sd']  = js['d_med'] / js['sd']
for name, col in [('median / MAD', 'cd_mad'), ('mean / sd', 'cd_sd')]:
    bb = [law_ruler(g, col)[1] for _, g in js.groupby('clip_factor')]
    print(f"  {name:14s} b by clip {[round(x,4) for x in bb]}   spread {max(bb)/min(bb):.2f}x")

a, b, r2, n = fit_two_term_law(threshold_cells(js[js['clip_factor'] == 12], ['alpha', 'k']))
print(f"\nthe Jump law the slide quotes (clip 12, penalty 100): "
      f"{a:.3f} + {b:.4f}*kappa   R2={r2:.3f}  (n={n} cells)   [slide: 0.312 + 0.0594, R2=0.987 -- MATCHES]")

print("\n*** DISCREPANCY -- this slide's other figures do NOT reproduce here ***")
print("  slide says: R2 0.977 (MAD) vs 0.736 (sd);  clip-invariance 1.03x (MAD) vs 1.36x (sd)")
print("  computed  : R2 as printed above;           clip-invariance as printed above")
print("  The Spearman rankings DO match (0.87 / 0.97 / 0.90). The sd-ruler here is")
print("  d_mean/sd on the 1,760 sweep and d_med/sd on jump_sweep (no d_mean column there),")
print("  so the original construction may differ. Treat those two slide figures as unverified.")


In [ ]:
# === `jump_penalty` - the price of a regime change ===
# Slide quotes: the bac / threshold table over the five penalties; lambda->0 == KMeans;
#               the optimum is 30 in all eleven slices; paired 30-vs-100 test;
#               and the two crossing-drop modes that make the gap conservative.
j = JUMP_SWEEP[JUMP_SWEEP['model'] == 'jump']
tab = j.groupby('jump_penalty')['bac'].mean()
thr = (threshold_cells(j, ['clip_factor', 'jump_penalty', 'alpha', 'k'])
       .groupby('jump_penalty')['thr'].mean())
print("penalty:      " + "  ".join(f"{p:>8g}" for p in tab.index))
print("mean bac:     " + "  ".join(f"{v:8.4f}" for v in tab.values))
print("mean threshold" + "  ".join(f"{thr.get(p, np.nan):8.4f}" for p in tab.index))
print(f"\nbest = {tab.idxmax():g};  production default 100 costs {tab[100.0]-tab.max():+.4f} bac")

slices, n_slices = [], 0
for by in ('k', 'alpha', 'clip_factor'):
    best = j.groupby([by, 'jump_penalty'])['bac'].mean().unstack().idxmax(axis=1)
    n_slices += len(best)
    slices.append(f"{by}({len(best)}): " + ("all->30" if set(best) == {30.0}
                  else ",".join(f"{i}->{v:g}" for i, v in best.items())))
print(f"\nis the optimum invariant?  best penalty per slice:   " + "   ".join(slices)
      + f"   -> {n_slices} slices, all 30")

key = ['loc', 'alpha', 'k', 'clip_factor', 'rep']
w = j.pivot_table(index=key, columns='jump_penalty', values='bac')
print(f"\npaired on identical draws (n={len(w)}):")
for p in (0.03, 0.3, 3.0, 100.0):
    d = w[30.0] - w[p]
    print(f"  penalty 30 vs {p:<6g} {d.mean():+.4f} bac   t={d.mean()/(d.std()/np.sqrt(len(d))):+7.1f}"
          f"   wins {100*(d>0).mean():.0f}% of draws")

km = JUMP_SWEEP[JUMP_SWEEP['model'] == 'kmeans'].set_index(key)['bac']
parts = []
for p in (0.03, 0.3, 3.0, 30.0, 100.0):
    s = j[j['jump_penalty'] == p].set_index(key)['bac']
    a_, b_ = s.align(km, join='inner')
    parts.append(f"{p:g}: d={(a_-b_).abs().mean():.4f} r={a_.corr(b_):.4f}")
print("\nlambda->0 the DP decouples and jump BECOMES KMeans (mean|diff|, corr):\n  " + "   ".join(parts))

# crossing() returns None from BOTH ends, and the two biases point opposite ways:
#   'ALREADY ABOVE' removes a config's EASIEST cells (its lowest thresholds) -> biases it UP
#   'never reaches' removes its HARDEST cells                                -> biases it DOWN
# Penalty 30 is the only setting losing cells for being too good, and 100 the one losing
# most for being too hard, so both effects flatter 100: the measured gap is a floor.
print("\nwhy the 30-vs-100 gap is conservative -- both drop modes favour 100:")
rows = []
for (cl, pen, a, k), g in j.groupby(['clip_factor', 'jump_penalty', 'alpha', 'k']):
    c = g.groupby('loc')['bac'].mean().sort_index()
    mode = ('never reaches' if c.max() < TARGET else
            'ALREADY ABOVE' if c.min() >= TARGET else 'crossed')
    rows.append(dict(penalty=pen, mode=mode))
print(pd.DataFrame(rows).pivot_table(index='penalty', columns='mode',
                                     aggfunc=len, values='mode').fillna(0).astype(int).to_string())


In [ ]:
# === KMeans++ - never a capability problem ===
# Slide quotes: p(bac>0.65), mean and max per feature set; the scale control;
#               and that clip only bites while a scale axis survives.
g = KM_SWEEP.groupby('feature_set')['bac']
tab = pd.DataFrame({'p(bac>0.65)': g.apply(lambda s: (s > 0.65).mean()),
                    'mean': g.mean(), 'max': g.max(), 'fits': g.size()})
print(tab.round(4).to_string())
print(f"\nscale control: {len(KM_SWEEP[KM_SWEEP['feature_set']=='scales'])} fits, "
      f"none above {KM_SWEEP[KM_SWEEP['feature_set']=='scales']['bac'].max():.3f}")

print("\np(bac>0.65) by clip:")
print("  *** DISCREPANCY: the slide says all-15 runs 0.35 -> 0.95 across clip 10->20.")
print("      Pooled over loc it is FLAT (below). The widest clip span anywhere is at")
print("      (loc 0.0018, alpha 0.8, k 2.0) and runs 1.00 -> 0.00, i.e. the other way.")
print("      Treat the slide's 0.35 -> 0.95 as unverified against this CSV. ***")
p = (KM_SWEEP.groupby(['feature_set', 'clip_factor'])['bac']
     .apply(lambda s: (s > 0.65).mean()).unstack())
print(p.round(3).to_string())

mm = KM_SWEEP[KM_SWEEP['feature_set'] == 'means']
byloc = mm.groupby('loc')['bac'].apply(lambda s: (s > 0.65).mean())
sat = [l for l, v in byloc.items() if v >= 0.999]
print("\nrestricted to the six means, p(bac>0.65) by loc:   "
      + "   ".join(f"{l}: {v:.3f}" for l, v in byloc.items() if v < 0.999)
      + f"   |   {min(sat)} and above: 1.000 (all {len(sat)} loc values)")


In [ ]:
# === The three laws, side by side - one sweep, one draw per cell ===
# Slide quotes: the three (a, b, R2) rows at clip 12.
c12 = threshold_cells(VAE_SWEEP[VAE_SWEEP['clip_factor'] == 12], ['model', 'alpha', 'k'])
LAWS = {}
print(f"  {'model':10s} {'a':>7s} {'b':>8s} {'R2':>6s} {'kappa max':>10s}")
for m in ('vae', 'jump', 'kmeans'):
    s = c12[c12['model'] == m]
    a, b, r2, n = fit_two_term_law(s); LAWS[m] = (a, b)
    print(f"  {m:10s} {a:7.3f} {b:8.4f} {r2:6.2f} {s['kappa'].max():10.2f}")
# a = what a model needs when tails are Gaussian (kappa=0); b = what each unit of excess
# kurtosis costs it.  A model can win on one and lose on the other -- which is exactly why
# the ranking is not fixed, and why the next cell finds where the lines cross.
print("\nthe VAE trades the worst intercept for the shallowest slope:")
print(f"  VAE slope is {100*(1 - LAWS['vae'][1]/np.mean([LAWS['jump'][1], LAWS['kmeans'][1]])):.0f}%"
      f" under the Jump/KMeans pair")


In [ ]:
# === So the ranking is not fixed - it crosses ===
# Slide quotes: required distance at kappa = 0, 3, 5, 8 and the two crossover points.
print(f"  {'kappa':>6s} {'VAE':>8s} {'Jump':>8s} {'KMeans':>8s}   needs least")
for kap in (0, 3, 5, 8):
    v = {m: LAWS[m][0] + LAWS[m][1] * kap for m in LAWS}
    best = min(v, key=v.get)
    print(f"  {kap:6d} {v['vae']:8.3f} {v['jump']:8.3f} {v['kmeans']:8.3f}   {best}")
for other in ('jump', 'kmeans'):
    da = LAWS['vae'][0] - LAWS[other][0]
    db = LAWS['vae'][1] - LAWS[other][1]
    print(f"\nVAE overtakes {other} at kappa = {-da/db:.2f}")


In [ ]:
# === What the VAE actually buys - not accuracy, but a floor ===
# Slide quotes: mean bac, sd, p(>0.6), p(<0.55), IQR per model; and that the
#               "ranking reverses at k=4" claim does not survive the full grid.
#
# bac = balanced accuracy; 0.5 is chance for two states.  So p(<0.55) is the share of the
# grid where a model is essentially NOT WORKING -- the failure rate.  The means here are
# within two points of each other, so what separates these models is not where they peak
# but how often they collapse; sd and IQR say the same thing about spread.
rows = []
for m, g in VAE_SWEEP.groupby('model'):
    b = g['bac']
    rows.append(dict(model=m, fits=len(b), mean=b.mean(), sd=b.std(),
                     **{'p(>0.6)': (b > 0.6).mean(), 'p(<0.55)': (b < 0.55).mean()},
                     IQR=b.quantile(0.75) - b.quantile(0.25)))
t = pd.DataFrame(rows).set_index('model').loc[['vae', 'kmeans', 'jump']]
print(t.round(4).to_string())
print(f"\nfailure rate: VAE {t.loc['vae','p(<0.55)']:.0%} vs baselines "
      f"{t.loc['kmeans','p(<0.55)']:.0%}-{t.loc['jump','p(<0.55)']:.0%};"
      f" IQR {100*(1-t.loc['vae','IQR']/t.loc['jump','IQR']):.0f}% narrower")
print("\nmean bac by k at clip 12 -- the reversal was an artifact of reading one loc:")
print(VAE_SWEEP[VAE_SWEEP['clip_factor'] == 12]
      .groupby(['k', 'model'])['bac'].mean().unstack().round(3).to_string())


In [ ]:
# === Provenance - which sweep backs which claim ===
# Slide quotes: fits, models x reps, swept axes and pinned config for each CSV.
import os
FILES = ['../data/vae_sweep.csv', '../data/vae_sweep_clip18.csv', '../data/jump_sweep.csv',
         '../data/kmeans_sweep.csv', '../data/jump_loc_sweep.csv', '../data/jump_moments.csv']
AX = ['loc', 'alpha', 'k', 'clip_factor', 'jump_penalty', 'feature_set']
for f in FILES:
    d = pd.read_csv(f)
    models = sorted(d['model'].unique()) if 'model' in d else ['jump (pre-dates the column)']
    swept = {c: len(d[c].dropna().unique()) for c in AX if c in d and d[c].nunique() > 1}
    pinned = {c: d[c].dropna().unique()[0] for c in AX if c in d and d[c].nunique() == 1}
    print(f"{os.path.basename(f):24s} {len(d):6d} rows  reps={d['rep'].nunique():2d}  "
          f"{','.join(models):22s} swept {swept}")
    print(f"{'':24s} pinned {pinned if pinned else '{}'}"
          f"   (+ always fixed: lamda_t=4.0, vae_epochs=500, stay_prob=0.96)")
print("\njump_sweep.csv performed 38,400 fits; KMeans ignores jump_penalty and was refit")
print("identically 5x per draw, so 15,360 duplicate rows were removed and one copy kept.")
